# Swiss Municipalities with Population over years

This code 

1. imports the Excel file *"Bilanz der ständigen Wohnbevölkerung nach Bezirken und Gemeinden, 1991-2025"* from [BFS](https://www.bfs.admin.ch/bfs/rm/home.assetdetail.36681837.html),
2. extracts municipality, bezirk, and canton from the first column, and
3. stores the result into a Parquet file and Excel

In [1]:
import polars as pl
import requests

## Download from website

Either automatically using the snippet below or manually.

In [2]:
url = "https://dam-api.bfs.admin.ch/hub/api/dam/assets/36681837/master"
resp = requests.get(url, timeout=30)
resp.raise_for_status()

with open("bfs_data.xlsx", "wb") as f:
    f.write(resp.content)

## Read specified sheets/years

In [3]:
# Adapt accordingly
last_year = 2025
first_year = 2011  # if you need <2011, adapt parser of first column

In [4]:
cols = [
    "region_raw",
    "population_1_jan",
    "births",
    "deaths",
    "birth_surplus",
    "in_migration",
    "out_migration",
    "net_migration",
    "adjustments",
    "population_31_dec",
    "change_absolute",
    "change_percent",
]

df_list = [
    pl.read_excel(
        "bfs_data.xlsx",
        sheet_name=str(year),
        has_header=False,
        read_options={"skip_rows": 4, "column_names": cols},
    ).with_columns(year=year)
    for year in range(last_year, first_year, -1)
]
df0 = pl.concat(df_list)

print("Data shape:", df0.shape)
df0.head()

Data shape: (33765, 13)


region_raw,population_1_jan,births,deaths,birth_surplus,in_migration,out_migration,net_migration,adjustments,population_31_dec,change_absolute,change_percent,year
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i32
"""Schweiz 2""",9051029,78153,71904,6249,715462,645030,70432,-585,9127125,76096,0.840744,2025
"""- Zürich""",1620020,14981,11654,3327,126748,116603,10145,-574,1632918,12898,0.796163,2025
""">> Bezirk Affoltern""",58190,455,380,75,4396,4083,313,-12,58566,376,0.646159,2025
"""......0001 Aeugst am Albis""",1995,11,15,-4,153,134,19,-7,2003,8,0.401003,2025
"""......0002 Affoltern am Albis""",12882,112,98,14,1118,1104,14,1,12911,29,0.22512,2025


## Parse first column into bfs_nr/municipality/bezirk/canton

In [5]:
canton_codes = {
    "Aargau": "AG",
    "Appenzell Ausserrhoden": "AR",
    "Appenzell Innerrhoden": "AI",
    "Basel-Landschaft": "BL",
    "Basel-Stadt": "BS",
    "Bern / Berne": "BE",
    "Fribourg / Freiburg": "FR",
    "Genève": "GE",
    "Glarus": "GL",
    "Graubünden / Grigioni / Grischun": "GR",
    "Jura": "JU",
    "Luzern": "LU",
    "Neuchâtel": "NE",
    "Nidwalden": "NW",
    "Obwalden": "OW",
    "Schaffhausen": "SH",
    "Schwyz": "SZ",
    "Solothurn": "SO",
    "St. Gallen": "SG",
    "Thurgau": "TG",
    "Ticino": "TI",
    "Uri": "UR",
    "Valais / Wallis": "VS",
    "Vaud": "VD",
    "Zug": "ZG",
    "Zürich": "ZH",
}


In [6]:
# 2. Regex patterns based on the BFS structural prefixes
regex_municipality = r"^\.{6}(\d{4})\s+(.*)$"
regex_bezirk = r"^>>\s*(.*)$"

df1 = (
    df0
    # Step A: Identify Canton and Bezirk rows, clean them, and push them into columns
    .with_columns(
        [
            # If line starts with "- ", extract the Canton name, else keep null
            pl.when(pl.col("region_raw").str.starts_with("- "))
            .then(pl.col("region_raw").str.replace(r"^-\s*", ""))
            .otherwise(None)
            .alias("canton"),
            # If line starts with ">> ", extract the Bezirk name, else keep null
            pl.when(pl.col("region_raw").str.starts_with(">> "))
            .then(pl.col("region_raw").str.replace(r"^>>\s*", ""))
            .otherwise(None)
            .alias("bezirk"),
        ]
    )
    # Forward-fill the Canton and Bezirk columns down through the nulls
    .with_columns([pl.col("canton").forward_fill(), pl.col("bezirk").forward_fill()])
    # Filter the rows to keep ONLY actual municipalities
    .filter(pl.col("region_raw").str.starts_with("......"))
    # Extract BFS Number and clean Municipality Name
    .with_columns(
        bfs_nr=pl.col("region_raw").str.extract(regex_municipality, 1).cast(pl.Int32),
        municipality=pl.col("region_raw").str.extract(regex_municipality, 2),
    )
    # Map cantons
    .with_columns(pl.col("canton").replace_strict(canton_codes, default=None))
    .select("year", "bfs_nr", "municipality", "bezirk", "canton", *cols)
    .drop("region_raw")
)
df1.head()

year,bfs_nr,municipality,bezirk,canton,population_1_jan,births,deaths,birth_surplus,in_migration,out_migration,net_migration,adjustments,population_31_dec,change_absolute,change_percent
i32,i32,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64
2025,1,"""Aeugst am Albis""","""Bezirk Affoltern""","""ZH""",1995,11,15,-4,153,134,19,-7,2003,8,0.401003
2025,2,"""Affoltern am Albis""","""Bezirk Affoltern""","""ZH""",12882,112,98,14,1118,1104,14,1,12911,29,0.22512
2025,3,"""Bonstetten""","""Bezirk Affoltern""","""ZH""",5712,40,43,-3,360,404,-44,-6,5659,-53,-0.927871
2025,4,"""Hausen am Albis""","""Bezirk Affoltern""","""ZH""",3951,31,29,2,222,221,1,2,3956,5,0.12655
2025,5,"""Hedingen""","""Bezirk Affoltern""","""ZH""",4002,26,21,5,325,354,-29,5,3983,-19,-0.474763


In [7]:
df1.filter(pl.col("municipality") == "Zürich").head()

year,bfs_nr,municipality,bezirk,canton,population_1_jan,births,deaths,birth_surplus,in_migration,out_migration,net_migration,adjustments,population_31_dec,change_absolute,change_percent
i32,i32,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64
2025,261,"""Zürich""","""Bezirk Zürich""","""ZH""",436551,4575,3096,1479,34779,31275,3504,-599,440935,4384,1.004235
2024,261,"""Zürich""","""Bezirk Zürich""","""ZH""",433989,4379,3130,1249,33906,31703,2203,-890,436551,2562,0.590338
2023,261,"""Zürich""","""Bezirk Zürich""","""ZH""",427721,4544,3239,1305,37559,31723,5836,-873,433989,6268,1.465441
2022,261,"""Zürich""","""Bezirk Zürich""","""ZH""",423193,4554,3428,1126,35590,31126,4464,-1062,427721,4528,1.069961
2021,261,"""Zürich""","""Bezirk Zürich""","""ZH""",421878,5233,3303,1930,33212,32748,464,-1079,423193,1315,0.311701


## Save as Parquet and Excel

In [8]:
df1.write_parquet("municipality.parquet")

In [9]:
df1.write_excel("municipality.xlsx")